In [1]:
import time
import requests
import pandas as pd
from tqdm.auto import tqdm

In [2]:
shots_2526 = pd.read_csv("data/shots_2526.csv")
shots_2526.head(10)

,shotID,arenaAdjustedShotDistance,arenaAdjustedXCord,arenaAdjustedXCordABS,arenaAdjustedYCord,arenaAdjustedYCordAbs,averageRestDifference,awayEmptyNet,awayPenalty1Length,awayPenalty1TimeLeft,...,xCordAdjusted,xFroze,xGoal,xPlayContinuedInZone,xPlayContinuedOutsideZone,xPlayStopped,xRebound,xShotWasOnGoal,yCord,yCordAdjusted
0,0,38.000000,-58.0,58.0,-22.0,22.0,0.0,0,0,0,...,58,0.308458,0.017510,0.350448,0.253295,0.035563,0.034727,0.791506,-22,22
1,1,59.000000,-33.0,33.0,-19.0,19.0,0.0,0,0,0,...,33,0.304403,0.036720,0.356975,0.235788,0.025301,0.040814,0.738728,-19,19
2,2,45.000000,55.0,55.0,-29.0,29.0,0.0,0,0,0,...,55,0.306225,0.016030,0.358237,0.263372,0.026669,0.029466,0.831215,-29,-29
3,3,63.000000,35.0,35.0,33.0,33.0,0.0,0,0,0,...,35,0.200191,0.009735,0.439703,0.294931,0.020398,0.035042,0.659939,33,33
4,4,68.000000,30.0,30.0,-34.0,34.0,0.0,0,0,0,...,30,0.219360,0.009977,0.422430,0.307621,0.020837,0.019774,0.681355,-34,-34
5,5,42.000000,-58.0,58.0,28.0,28.0,7.4,0,0,0,...,58,0.332180,0.049217,0.300620,0.245800,0.033867,0.038316,0.767188,28,-28
6,6,44.000000,-47.0,47.0,-12.0,12.0,0.0,0,0,0,...,47,0.233335,0.016412,0.376547,0.284262,0.038632,0.050812,0.801125,-12,12
7,7,13.000000,-79.0,79.0,9.0,9.0,-11.0,0,0,0,...,79,0.113928,0.085066,0.393169,0.298180,0.027855,0.081801,0.808193,9,-9
8,8,8.062258,-82.0,82.0,4.0,4.0,-8.8,0,0,0,...,82,0.141561,0.081322,0.390513,0.296467,0.024592,0.065546,0.748913,4,-4
9,9,6.082763,-88.0,88.0,6.0,6.0,-8.8,0,0,0,...,88,0.225088,0.231689,0.255184,0.146837,0.042034,0.099168,0.817503,6,-6


In [3]:
# NHL-style game ID
shots_2526["nhl_game_id"] = (
    shots_2526["season"].astype(int) * 1_000_000
    + shots_2526["game_id"].astype(int)
)


games_df = (
    shots_2526.loc[
        shots_2526["isPlayoffGame"] == 0,
        [
            "nhl_game_id",
            "game_id",
            "season",
            "homeTeamCode",
            "awayTeamCode"
        ]
    ]
    .drop_duplicates(subset="nhl_game_id")
    .sort_values("nhl_game_id")
    .reset_index(drop=True)
)

games_df.head()

,nhl_game_id,game_id,season,homeTeamCode,awayTeamCode
0,2025020001,20001,2025,FLA,CHI
1,2025020002,20002,2025,NYR,PIT
2,2025020003,20003,2025,LAK,COL
3,2025020004,20004,2025,TOR,MTL
4,2025020005,20005,2025,WSH,BOS


In [4]:

all_shifts = []

game_ids = games_df["nhl_game_id"].unique()

for game_id in tqdm(game_ids, desc="Downloading NHL shifts"):

    url = (
        "https://api.nhle.com/stats/rest/en/shiftcharts"
        f"?cayenneExp=gameId={game_id}"
    )

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    game_shifts = pd.DataFrame(data["data"])
    all_shifts.append(game_shifts)

    time.sleep(0.1)

shifts_df = pd.concat(all_shifts, ignore_index=True)

shifts_df.head()

,id,detailCode,duration,endTime,eventDescription,eventDetails,eventNumber,firstName,gameId,hexValue,lastName,period,playerId,shiftNumber,startTime,teamAbbrev,teamId,teamName,typeCode
0,15536563,0.0,00:34,00:34,None,None,6.0,Brad,2025020001,#041E42,Marchand,1,8473419,1,00:00,FLA,13,Florida Panthers,517
1,15536606,0.0,01:15,04:18,None,None,154.0,Brad,2025020001,#041E42,Marchand,1,8473419,2,03:03,FLA,13,Florida Panthers,517
2,15536607,0.0,00:19,06:35,None,None,207.0,Brad,2025020001,#041E42,Marchand,1,8473419,3,06:16,FLA,13,Florida Panthers,517
3,15536608,0.0,01:02,08:00,None,None,216.0,Brad,2025020001,#041E42,Marchand,1,8473419,4,06:58,FLA,13,Florida Panthers,517
4,15536609,0.0,00:26,10:03,None,None,255.0,Brad,2025020001,#041E42,Marchand,1,8473419,5,09:37,FLA,13,Florida Panthers,517


In [5]:
shifts_df.shape

(992876, 19)

In [6]:
shifts_df.to_csv(
    "data/shifts_2526.csv",
    index=False
)

### Inspect Raw Shift Data

Examine the structure and contents of the NHL shift data before performing any preprocessing.

The shift records will ultimately be used to identify the players on the ice during each interval of play and construct no-substitution stints for the RAPM models.

In [7]:
shifts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 992876 entries, 0 to 992875
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                992876 non-null  int64  
 1   detailCode        992844 non-null  float64
 2   duration          984474 non-null  object 
 3   endTime           992876 non-null  object 
 4   eventDescription  8402 non-null    object 
 5   eventDetails      8402 non-null    object 
 6   eventNumber       992605 non-null  float64
 7   firstName         992876 non-null  object 
 8   gameId            992876 non-null  int64  
 9   hexValue          959977 non-null  object 
 10  lastName          992876 non-null  object 
 11  period            992876 non-null  int64  
 12  playerId          992876 non-null  int64  
 13  shiftNumber       992876 non-null  int64  
 14  startTime         992876 non-null  object 
 15  teamAbbrev        992876 non-null  object 
 16  teamId            99

In [8]:
shifts_df[
    [
        "gameId",
        "period",
        "playerId",
        "firstName",
        "lastName",
        "teamAbbrev",
        "shiftNumber",
        "startTime",
        "endTime",
        "duration"
    ]
].head(20)

,gameId,period,playerId,firstName,lastName,teamAbbrev,shiftNumber,startTime,endTime,duration
0,2025020001,1,8473419,Brad,Marchand,FLA,1,00:00,00:34,00:34
1,2025020001,1,8473419,Brad,Marchand,FLA,2,03:03,04:18,01:15
2,2025020001,1,8473419,Brad,Marchand,FLA,3,06:16,06:35,00:19
3,2025020001,1,8473419,Brad,Marchand,FLA,4,06:58,08:00,01:02
4,2025020001,1,8473419,Brad,Marchand,FLA,5,09:37,10:03,00:26
5,2025020001,1,8473419,Brad,Marchand,FLA,6,11:54,13:48,01:54
6,2025020001,1,8473419,Brad,Marchand,FLA,7,15:17,15:47,00:30
7,2025020001,1,8473419,Brad,Marchand,FLA,8,17:37,18:24,00:47
8,2025020001,1,8473419,Brad,Marchand,FLA,9,19:47,20:00,00:13
9,2025020001,2,8473419,Brad,Marchand,FLA,10,01:09,01:12,00:03


In [9]:
shifts_df["period"].value_counts().sort_index()

period
1    340892
2    310493
3    331789
4      9437
5       265
Name: count, dtype: int64

### Examine Period 5 Shift Records

Inspect shift records assigned to period 5.

Because the RAPM models are based on continuous periods of play, these records should be examined before determining whether they should be retained or excluded from stint construction.

In [10]:
shifts_df.loc[
    shifts_df["period"] == 5,
    [
        "gameId",
        "period",
        "playerId",
        "firstName",
        "lastName",
        "teamAbbrev",
        "shiftNumber",
        "startTime",
        "endTime",
        "duration"
    ]
].head(20)

,gameId,period,playerId,firstName,lastName,teamAbbrev,shiftNumber,startTime,endTime,duration
3978,2025020006,5,8475172,Nazem,Kadri,CGY,0,00:00,00:00,None
4203,2025020006,5,8477934,Leon,Draisaitl,EDM,0,00:00,00:00,None
4393,2025020006,5,8480028,Morgan,Frost,CGY,0,00:00,00:00,None
5129,2025020007,5,8477960,Adrian,Kempe,LAK,0,00:00,00:00,None
5334,2025020007,5,8479675,Trevor,Moore,LAK,0,00:00,00:00,None
5446,2025020007,5,8481604,Pavel,Dorofeyev,VGK,0,00:00,00:00,None
26073,2025020034,5,8477501,Valeri,Nichushkin,COL,0,00:00,00:00,None
26155,2025020034,5,8478420,Mikko,Rantanen,DAL,0,00:00,00:00,None
26273,2025020034,5,8480027,Jason,Robertson,DAL,0,00:00,00:00,None
36363,2025020047,5,8482079,Marco,Rossi,MIN,0,00:00,00:00,None


In [11]:
print(
    "Period 5 records:",
    (shifts_df["period"] == 5).sum()
)

print(
    "Games with period 5 records:",
    shifts_df.loc[
        shifts_df["period"] == 5,
        "gameId"
    ].nunique()
)

Period 5 records: 265
Games with period 5 records: 119


### Remove Shootout Records

Exclude period 5 records from the shift data.

Period 5 observations have no recorded shift duration and represent shootout participation rather than continuous on-ice shifts. Regulation periods (1–3) and overtime (period 4) are retained for stint construction.

In [12]:
shifts_df = (
    shifts_df.loc[
        shifts_df["period"].isin([1, 2, 3, 4])
    ]
    .copy()
    .reset_index(drop=True)
)

shifts_df["period"].value_counts().sort_index()

period
1    340892
2    310493
3    331789
4      9437
Name: count, dtype: int64

### Convert Shift Times to Period-Relative Seconds

Convert shift start and end times from `MM:SS` format to elapsed seconds within each period.

This places the shift data on the same period-relative time scale as `shots_df["period_time"]`, allowing shot events to be matched to the appropriate on-ice stint.

In [13]:
# Start time of each period in cumulative game seconds
period_start = {
    1: 0,
    2: 1200,
    3: 2400,
    4: 3600
}

# Convert MM:SS to seconds
def time_to_seconds(time_str):
    minutes, seconds = map(int, time_str.split(":"))
    return minutes * 60 + seconds


# Convert shift times to cumulative game seconds
shifts_df["start_time"] = (
    shifts_df["startTime"].apply(time_to_seconds)
    + shifts_df["period"].map(period_start)
)

shifts_df["end_time"] = (
    shifts_df["endTime"].apply(time_to_seconds)
    + shifts_df["period"].map(period_start)
)

# Calculate elapsed time within the period
shifts_df["period_start_time"] = (
    shifts_df["start_time"]
    - shifts_df["period"].map(period_start)
)

shifts_df["period_end_time"] = (
    shifts_df["end_time"]
    - shifts_df["period"].map(period_start)
)

In [14]:
shifts_df[
    [
        "period",
        "startTime",
        "endTime",
        "start_time",
        "end_time",
        "period_start_time",
        "period_end_time"
    ]
].head(20)

,period,startTime,endTime,start_time,end_time,period_start_time,period_end_time
0,1,00:00,00:34,0,34,0,34
1,1,03:03,04:18,183,258,183,258
2,1,06:16,06:35,376,395,376,395
3,1,06:58,08:00,418,480,418,480
4,1,09:37,10:03,577,603,577,603
5,1,11:54,13:48,714,828,714,828
6,1,15:17,15:47,917,947,917,947
7,1,17:37,18:24,1057,1104,1057,1104
8,1,19:47,20:00,1187,1200,1187,1200
9,2,01:09,01:12,1269,1272,69,72


In [15]:
shift_time_check = (
    shifts_df
    .groupby("period")
    .agg(
        min_period_start=("period_start_time", "min"),
        max_period_end=("period_end_time", "max"),
        min_game_start=("start_time", "min"),
        max_game_end=("end_time", "max")
    )
)

shift_time_check

,min_period_start,max_period_end,min_game_start,max_game_end
period,,,,
1,0,1200,0,1200
2,0,1200,1200,2400
3,0,1200,2400,3600
4,0,300,3600,3900


### Check for Duplicate Shift Records

Check whether identical player shifts appear more than once in the NHL shift data.

Duplicate shift records could affect lineup reconstruction and stint construction, so they should be identified before further preprocessing.

In [16]:
duplicate_cols = [
    "gameId",
    "playerId",
    "period",
    "shiftNumber",
    "period_start_time",
    "period_end_time"
]

shifts_df.duplicated(
    subset=duplicate_cols
).sum()

832

In [17]:
duplicate_shifts = (
    shifts_df.loc[
        shifts_df.duplicated(
            subset=duplicate_cols,
            keep=False
        )
    ]
    .sort_values(
        [
            "gameId",
            "playerId",
            "period",
            "shiftNumber",
            "period_start_time"
        ]
    )
)

duplicate_shifts.head(30)

,id,detailCode,duration,endTime,eventDescription,eventDetails,eventNumber,firstName,gameId,hexValue,...,shiftNumber,startTime,teamAbbrev,teamId,teamName,typeCode,start_time,end_time,period_start_time,period_end_time
4835,15541102,0.0,00:44,02:10,None,None,507.0,Brian,2025020007,#111111,...,9,01:26,LAK,26,Los Angeles Kings,517,1286,1330,86,130
4836,15541405,0.0,00:44,02:10,None,None,507.0,Brian,2025020007,#111111,...,9,01:26,LAK,26,Los Angeles Kings,517,1286,1330,86,130
4968,15541109,0.0,00:42,02:10,None,None,509.0,Phillip,2025020007,#111111,...,9,01:28,LAK,26,Los Angeles Kings,517,1288,1330,88,130
4969,15541413,0.0,00:42,02:10,None,None,509.0,Phillip,2025020007,#111111,...,9,01:28,LAK,26,Los Angeles Kings,517,1288,1330,88,130
4991,15541110,0.0,00:43,02:10,None,None,508.0,Cody,2025020007,#111111,...,8,01:27,LAK,26,Los Angeles Kings,517,1287,1330,87,130
4992,15541414,0.0,00:43,02:10,None,None,508.0,Cody,2025020007,#111111,...,8,01:27,LAK,26,Los Angeles Kings,517,1287,1330,87,130
5340,15541126,0.0,00:47,00:47,None,None,442.0,Mikey,2025020007,#111111,...,11,00:00,LAK,26,Los Angeles Kings,517,1200,1247,0,47
5341,15541439,0.0,00:47,00:47,None,None,442.0,Mikey,2025020007,#111111,...,11,00:00,LAK,26,Los Angeles Kings,517,1200,1247,0,47
5451,15541130,0.0,00:48,00:48,None,None,442.0,Quinton,2025020007,#111111,...,10,00:00,LAK,26,Los Angeles Kings,517,1200,1248,0,48
5452,15541448,0.0,00:48,00:48,None,None,442.0,Quinton,2025020007,#111111,...,10,00:00,LAK,26,Los Angeles Kings,517,1200,1248,0,48


In [18]:
print("Rows involved:", len(duplicate_shifts))

print(
    "Unique duplicate groups:",
    duplicate_shifts[
        duplicate_cols
    ].drop_duplicates().shape[0]
)

Rows involved: 1664
Unique duplicate groups: 832


### Remove Duplicate Shift Records

Duplicate shift records were identified using game, player, period, shift number, and shift start/end times.

A total of 832 duplicate groups were identified, containing 1,664 records. Inspection showed that each duplicate group contained two observations representing the same player shift, despite having different NHL shift-record identifiers.

One observation from each duplicate group is retained to prevent duplicate shift records from affecting lineup and stint construction.

In [19]:
shifts_df = (
    shifts_df
    .drop_duplicates(
        subset=duplicate_cols,
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "Remaining duplicate shifts:",
    shifts_df.duplicated(
        subset=duplicate_cols
    ).sum()
)

print("Remaining shift records:", len(shifts_df))

Remaining duplicate shifts: 0
Remaining shift records: 991779


### Validate Shift Durations

Compare the recorded NHL shift duration with the duration calculated from the shift start and end times.

This check helps identify malformed or inconsistent shift records before the shift data are used to reconstruct on-ice lineups and construct RAPM stints.

In [20]:
shifts_df["recorded_duration"] = (
    shifts_df["duration"]
    .apply(
        lambda x: time_to_seconds(x)
        if pd.notna(x)
        else pd.NA
    )
)

shifts_df["calculated_duration"] = (
    shifts_df["period_end_time"]
    - shifts_df["period_start_time"]
)

In [21]:
print(
    "Missing recorded durations:",
    shifts_df["recorded_duration"].isna().sum()
)

Missing recorded durations: 8138


In [22]:
valid_duration = shifts_df["recorded_duration"].notna()

duration_check = (
    shifts_df.loc[valid_duration, "recorded_duration"]
    - shifts_df.loc[valid_duration, "calculated_duration"]
)

print("Exact matches:", (duration_check == 0).sum())
print("Mismatches:", (duration_check != 0).sum())

Exact matches: 983624
Mismatches: 17


### Inspect Shift Duration Mismatches

Most recorded shift durations agree exactly with the duration calculated from the shift start and end times.

Records with missing NHL durations can still have their duration calculated from the shift boundaries. A small number of records contain disagreements between the recorded and calculated durations and are inspected before further preprocessing.

In [23]:
shifts_df.loc[
    shifts_df["recorded_duration"].notna()
    & (
        shifts_df["recorded_duration"]
        != shifts_df["calculated_duration"]
    ),
    [
        "gameId",
        "period",
        "playerId",
        "firstName",
        "lastName",
        "teamAbbrev",
        "startTime",
        "endTime",
        "duration",
        "recorded_duration",
        "calculated_duration"
    ]
]

,gameId,period,playerId,firstName,lastName,teamAbbrev,startTime,endTime,duration,recorded_duration,calculated_duration
16661,2025020022,1,8475311,Darcy,Kuemper,LAK,00:00,20:00,09:30,570,1200
16847,2025020022,1,8476945,Connor,Hellebuyck,WPG,00:00,20:00,09:30,570,1200
139118,2025020182,2,8481668,Arturs,Silovs,PIT,00:00,12:16,00:02,2,736
400365,2025020528,3,8475831,Philipp,Grubauer,SEA,00:00,10:38,02:49,169,638
467469,2025020616,4,8478916,Joey,Daccord,SEA,00:00,01:52,01:43,103,112
473085,2025020623,3,8481020,Justus,Annunen,NSH,00:00,19:51,12:05,725,1191
481586,2025020634,2,8483710,Dennis,Hildeby,TOR,00:00,13:31,04:43,283,811
765976,2025021012,3,8475311,Darcy,Kuemper,LAK,00:00,01:04,00:55,55,64
780041,2025021031,3,8475683,Sergei,Bobrovsky,FLA,00:00,03:56,03:52,232,236
780043,2025021031,4,8475683,Sergei,Bobrovsky,FLA,00:00,00:00,19:54,1194,0


### Goalie Shift Duration Anomalies

Seventeen shift records had recorded NHL durations that differed from the duration calculated using the shift start and end times.

Inspection showed that all 17 records belonged to goaltenders. This suggests that the NHL shift feed handles some goalie records differently from standard skater shifts, particularly around period boundaries and goalie substitutions.

Because these records are not duplicates and may contain useful goalie information, they are retained. Skaters and goaltenders will be identified separately before constructing the on-ice stints.

## Construct On-Ice Stints

Use the cleaned NHL shift data to reconstruct intervals of play during which the set of players on the ice remains unchanged.

A new stint begins whenever a player enters or leaves the ice. These no-substitution stints form the observational units used in the RAPM regressions.

Shot events will subsequently be assigned to the stint in which they occurred, allowing CF, xGF, and GF outcomes to be calculated for each stint.

In [24]:
shifts_df.to_csv(
    "data/shifts_2526_clean.csv",
    index=False
)